# Entrega E1 · Tema 3. GLM para clasificación

**Técnicas Avanzadas de Predicción en Negocios · 22-10-2026**

Datos: Encuesta Cuatrienal de Estructura Salarial 2022 (INE), microdatos. Cada alumno trabaja con una muestra propia de 10 000 trabajadores.

**Cómo está organizada la entrega**

- **Parte A. Código mostrado.** Ejecuta y lee cada celda. Es el patrón que usarás en la parte B.
- **Parte B. Actividad.** Cuatro bloques con tareas cortas de código que escribes tú, de principio a fin, y una interpretación al final de cada bloque.
- **Cierre.** Una celda guarda tus respuestas en un fichero `.json`. Subes el cuaderno y ese fichero.

Tienes **120 minutos**, incluidos la revisión y la subida. Orientación: 10 min para la preparación, 22 para la parte A, 20 para los bloques 1 y 2, 15 para el bloque 3, 10 para el bloque 4 y 15 para el cierre.

**Solo si trabajas en Google Colab.** La celda siguiente instala la versión de scikit-learn del entorno `tap` del curso. En tu ordenador, con el entorno `tap` activado, no hace nada. Si Colab pide reiniciar la sesión después de instalar, acepta y vuelve a ejecutar el cuaderno desde el principio.

In [ ]:
# Instala las versiones del curso (solo en Google Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q -r https://raw.githubusercontent.com/juanDmontoro/tap/main/requirements.txt

## 0. Tu ID

> **Escribe solo el ID de seis cifras que te ha dado el profesor.**
> **No escribas tu nombre, usuario, DNI, NIU ni correo.** El cuaderno no debe contener ningún dato personal.

Tu ID es también tu **semilla**. La semilla fija tu muestra de datos y tus particiones, así que cada alumno obtiene resultados distintos.

In [ ]:
import re

# Escribe entre las comillas el ID de seis cifras que te ha dado el profesor.
ID = "ESCRIBE TU ID"

# El ID es también tu semilla: un número entero.
ID = ID.strip()
assert re.fullmatch(r"\d{6}", ID), "Escribe tu ID de seis cifras entre las comillas."
SEMILLA = int(ID)
print("ID y semilla:", SEMILLA)

### Preparación

**Ejecuta la celda siguiente.** Carga las bibliotecas y los datos y conserva la población del tema 2 (salario positivo, con tope en el percentil 99). Después elige 10 000 trabajadores con tu semilla. No hace falta leerla. Tarda unos segundos porque descarga los datos.

In [ ]:
# Bibliotecas para tratar datos y dibujar.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

# Preparación de los predictores y modelos.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

# Divisiones, validación cruzada y búsqueda de hiperparámetros.
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict

# Métricas y matriz de confusión.
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    recall_score, precision_score, log_loss,
)

plt.rcParams.update({"figure.figsize": (5, 4), "axes.spines.top": False,
                     "axes.spines.right": False})

# Datos de la EES 2022 en la web del curso (separador: tabulación).
URL_DATOS = "https://juandmontoro.github.io/tap/data/EES_2022.tab"
df = pd.read_csv(URL_DATOS, sep="\t", low_memory=False)
df.columns = df.columns.str.replace('"', "").str.strip()

# Población del tema 2: salario positivo y hasta el percentil 99.
tope = df["RETRINOIN"].quantile(0.99)
salario_admitido = (df["RETRINOIN"] > 0) & (df["RETRINOIN"] <= tope)
limpio = df[salario_admitido].copy()

# Tu muestra: 10 000 trabajadores elegidos con tu semilla.
datos = limpio.sample(n=10000, random_state=SEMILLA).reset_index(drop=True)
print("Observaciones:", len(datos))
print("Versión de scikit-learn:", sklearn.__version__)

### Predictores y división entre entrenamiento y test

Usamos los predictores del tema 2: 14 categóricos y 2 numéricos. El salario bruto anual (`RETRINOIN`) **no** es predictor, porque con él se definen las dos respuestas de esta entrega. Reservamos el 25 % de las observaciones para el test, que en esta entrega no se usa.

In [ ]:
PRED_CATEGORICOS = [
    "SEXO", "ESTU", "ANOS2", "TIPOCON", "TIPOJOR", "CNO1", "CNACE",
    "CONTROL", "MERCADO", "REGULACION", "RESPONSA", "NUTS1", "TIPOPAIS", "ESTRATO2",
]
PRED_NUMERICOS = ["ANOANTI", "JAP"]

# X contiene los predictores e y, el salario.
X = datos[PRED_CATEGORICOS + PRED_NUMERICOS].copy()
X[PRED_CATEGORICOS] = X[PRED_CATEGORICOS].astype("category")
y = datos["RETRINOIN"]

# Separamos el test antes de ajustar ningún modelo.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA)
print("Entrenamiento:", len(X_train), "| test:", len(X_test))

El preprocesador es el del tema 2: escala las variables numéricas y crea columnas indicadoras para las categóricas. Cada llamada a `crear_preprocesador()` devuelve uno nuevo, sin ajustar.

In [ ]:
# Devuelve un preprocesador nuevo en cada llamada.
def crear_preprocesador():
    return ColumnTransformer([
        ("numericas", StandardScaler(), PRED_NUMERICOS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01,
                                      sparse_output=False), PRED_CATEGORICOS),
    ])

# Parte A. Código mostrado

Ejecuta y lee cada celda. En esta parte no hay preguntas: es el patrón que adaptarás en la parte B.

## A1. Logit binario: ¿salario alto?

La respuesta vale 1 (un **positivo**) si el salario supera el **percentil 75 del entrenamiento** y 0 en otro caso. Así, un 25 % de los trabajadores son positivos: su salario está en el 25 % más alto. El logit es un GLM con respuesta binaria y enlace logit: estima la probabilidad de salario alto.

Obtenemos la probabilidad de validación cruzada de cada trabajador con `cross_val_predict`, como en el tema 2. `StratifiedKFold` mantiene en cada partición la proporción de positivos.

> **Python que usamos aquí**
> - `cross_val_predict(modelo, X, respuesta, cv=particiones, method=...)` devuelve una predicción de validación cruzada para cada observación. El argumento `method` decide cuál. Con `method="predict_proba"` devuelve la probabilidad predicha de cada clase, una columna por clase. Con `method="predict"` devuelve la clase predicha, que es la de mayor probabilidad (en un modelo binario, la que supera 0,5).

In [ ]:
# Umbral del salario alto: percentil 75 del entrenamiento.
p75 = y_train.quantile(0.75)

# Respuesta binaria: 1 = salario alto, 0 = no.
alto_train = (y_train > p75).astype(int)
print("P75:", round(p75), "euros | proporción de positivos:", round(alto_train.mean(), 3))

# Cinco particiones que conservan la proporción de cada clase.
cv_estratificada = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

# Modelo completo: preprocesador seguido del logit.
logit = Pipeline([
    ("preprocesado", crear_preprocesador()),
    ("modelo", LogisticRegression(max_iter=2000)),
])

# Probabilidades de validación cruzada: una columna por clase.
probabilidades_cv = cross_val_predict(logit, X_train, alto_train, cv=cv_estratificada,
                                      method="predict_proba")
probabilidades_cv

`probabilidades_cv` tiene una fila por trabajador y dos columnas: la primera es la probabilidad de la clase 0 (negativo, salario no alto) y la segunda, la de la clase 1 (positivo, salario alto). En cada fila suman 1. Nos quedamos solo con la segunda.

In [ ]:
# [:, 1]: todas las filas de la columna 1, la probabilidad de salario alto.
prob_alto_cv = probabilidades_cv[:, 1]

Con el umbral 0,5 obtenemos la clase predicha, la matriz de confusión y las métricas del tema 2. La log-loss mide la calidad de las probabilidades: cuanto menor, mejor.

> **Python que usamos aquí**
> - `confusion_matrix(observado, predicho)` devuelve la matriz de confusión: filas = clase observada (0, 1), columnas = clase predicha (0, 1).
> - `ConfusionMatrixDisplay(matriz, display_labels=nombres).plot()` la dibuja, con los nombres de las clases en los ejes.
> - `recall_score(observado, predicho)` y `precision_score(observado, predicho)` devuelven la sensibilidad y la precisión.
> - `log_loss(observado, probabilidades_predichas)` admite el vector de la probabilidad de la clase 1 o la tabla con una columna por clase.

In [ ]:
# Clase predicha con el umbral 0,5: 1 si la probabilidad llega a 0,5 y 0 si no.
pred_05 = (prob_alto_cv >= 0.5).astype(int)

# Matriz de confusión: filas = observado, columnas = predicho.
matriz_05 = confusion_matrix(alto_train, pred_05)
ConfusionMatrixDisplay(matriz_05, display_labels=["bottom75", "top25"]).plot(cmap="Blues", colorbar=False)
plt.show()

# Casilla [fila 0, columna 1]: observados 0 que se predicen como 1, los falsos positivos.
print("Falsos positivos:", matriz_05[0, 1])

# Sensibilidad, precisión y log-loss.
sens_05 = recall_score(alto_train, pred_05)
prec_05 = precision_score(alto_train, pred_05)
logloss_05 = log_loss(alto_train, prob_alto_cv)
print("Sensibilidad:", round(sens_05, 3), "| precisión:", round(prec_05, 3), "| log-loss:", round(logloss_05, 3))

¿Son buenas esas cifras? Las comparamos con las del **clasificador de referencia**, que no usa los predictores: asigna a todos la misma probabilidad, la proporción de positivos de su entrenamiento. Lo evaluamos con las mismas particiones que el logit y, con la misma función `cross_val_predict`, obtenemos sus probabilidades (`method="predict_proba"`) y su clase predicha (`method="predict"`).

> **Python que usamos aquí**
> - `DummyClassifier(strategy="prior")` es el clasificador de referencia: ignora los predictores y predice para todas las filas la proporción de cada clase en el entrenamiento.

In [ ]:
# Clasificador de referencia: asigna a todos la proporción de positivos de su entrenamiento.
referencia = DummyClassifier(strategy="prior")

# Sus probabilidades y su clase predicha de validación cruzada, con las mismas particiones que el logit.
prob_referencia_cv = cross_val_predict(referencia, X_train, alto_train, cv=cv_estratificada,
                                       method="predict_proba")
pred_referencia = cross_val_predict(referencia, X_train, alto_train, cv=cv_estratificada,
                                    method="predict")

# Su sensibilidad y su log-loss.
sens_ref = recall_score(alto_train, pred_referencia)
logloss_ref = log_loss(alto_train, prob_referencia_cv)
print("Sensibilidad:", round(sens_05, 3), "| sensibilidad referencia:", round(sens_ref, 3))
print("Log-loss:", round(logloss_05, 3), "| log-loss referencia:", round(logloss_ref, 3))

**Cómo se lee.** La referencia da a todos la misma probabilidad de salario alto, la proporción de positivos (en torno a 0,25). Como no llega al umbral 0,5, no predice ningún positivo: su sensibilidad es 0, y su precisión no se puede calcular porque no hay predichos positivos. El logit, en cambio, detecta una parte de los salarios altos. La log-loss compara las probabilidades: la del logit es menor que la de la referencia porque los predictores separan a los trabajadores de salario alto de los demás.

## A2. Logit multinomial: tres tramos salariales

A continuación redefinimos la respuesta con tres tramos salariales para entrenar un modelo multiclase. Los tramos se definen con los **terciles del entrenamiento**: 0 = bajo, 1 = medio, 2 = alto, con un tercio de los trabajadores en cada uno. Con tres clases, `LogisticRegression` ajusta el **logit multinomial** sin cambiar ningún argumento: estima una probabilidad por tramo, y las tres suman 1.

> **Python que usamos aquí**
> - `pd.qcut(variable, q, labels=False)` discretiza una variable numérica en `q` clases con el mismo número de observaciones (cuantiles). Con `q=3`, terciles. Asigna a cada valor el código de su intervalo: 0, 1 o 2.

In [ ]:
# Tramo de cada trabajador: 0 = bajo, 1 = medio, 2 = alto.
tramo_train = pd.qcut(y_train, q=3, labels=False)
NOMBRES_TRAMOS = ["bajo", "medio", "alto"]

# Con tres clases, LogisticRegression ajusta el logit multinomial.
multinomial = Pipeline([
    ("preprocesado", crear_preprocesador()),
    ("modelo", LogisticRegression(max_iter=2000)),
])

# Probabilidades de validación cruzada: una columna por tramo (0, 1, 2).
prob_tramos_cv = cross_val_predict(multinomial, X_train, tramo_train, cv=cv_estratificada,
                                   method="predict_proba")

`prob_tramos_cv` tiene una fila por trabajador y tres columnas: la probabilidad predicha de cada tramo, $P(Y=0 \mid X)$, $P(Y=1 \mid X)$ y $P(Y=2 \mid X)$.

In [ ]:
# Una fila por trabajador y una columna por tramo.
prob_tramos_cv

El tramo predicho es el de mayor probabilidad. Lo obtenemos con la misma función, ahora con `method="predict"`:

In [ ]:
# Tramo predicho de validación cruzada: el de mayor probabilidad.
tramo_predicho = cross_val_predict(multinomial, X_train, tramo_train, cv=cv_estratificada,
                                   method="predict")
tramo_predicho

En la matriz 3×3, las filas son los tramos observados y las columnas, los predichos. La comparamos de nuevo con el clasificador de referencia, que ahora predice siempre el tramo más frecuente de su entrenamiento.

In [ ]:
# Matriz de confusión 3×3: filas = tramo observado, columnas = tramo predicho.
matriz_tramos = confusion_matrix(tramo_train, tramo_predicho)
ConfusionMatrixDisplay(matriz_tramos, display_labels=NOMBRES_TRAMOS).plot(cmap="Blues", colorbar=False)
plt.show()

# Exactitud del modelo y del clasificador de referencia, con las mismas particiones.
exactitud = accuracy_score(tramo_train, tramo_predicho)
tramo_referencia = cross_val_predict(referencia, X_train, tramo_train, cv=cv_estratificada,
                                     method="predict")
exactitud_ref = accuracy_score(tramo_train, tramo_referencia)
print("Exactitud:", round(exactitud, 3), "| exactitud referencia:", round(exactitud_ref, 3))

**Cómo se lee la matriz.**

- La **diagonal** recoge los aciertos. La exactitud es la suma de la diagonal dividida por el total.
- Cada **fila** suma los trabajadores de un tramo observado. Su casilla de la diagonal dividida por el total de la fila es la **sensibilidad** de ese tramo.
- Fuera de la diagonal están los errores. Las casillas contiguas a la diagonal confunden **tramos vecinos** (bajo con medio, medio con alto). Las dos esquinas confunden bajo con alto: son los errores **extremos**.

Con tres tramos del mismo tamaño, el clasificador de referencia acierta en torno a un tercio de las veces.

## A3. Penalización del multinomial con pocas observaciones

`LogisticRegression` penaliza la suma de los coeficientes al cuadrado, como Ridge. Su argumento `C` es el **inverso** de la intensidad de la penalización: un `C` pequeño penaliza mucho y un `C` grande casi nada. Con pocas observaciones, la penalización evita probabilidades extremas que fallan en validación.

Tomamos 300 observaciones del entrenamiento y buscamos `C` con `GridSearchCV`, con la log-loss como métrica.

> **Python que usamos aquí**
> - `train_test_split(X, respuesta, train_size=300, random_state=SEMILLA)` es la función de la preparación: con `train_size=300` devuelve los predictores y la respuesta de 300 observaciones elegidas al azar y, después, los del resto.

In [ ]:
# Submuestra de 300 observaciones del entrenamiento. El resto (_) no se usa.
X_300, _, tramo_300, _ = train_test_split(X_train, tramo_train, train_size=300, random_state=SEMILLA)

# Malla de valores de C (inverso de la penalización) entre 10**(-3) y 10**2.
MALLA_C = {"modelo__C": np.logspace(-3, 2, 11)}

# Cada C se evalúa con las mismas cinco particiones; métrica: log-loss.
busqueda_tramos = GridSearchCV(multinomial, MALLA_C, scoring="neg_log_loss", cv=cv_estratificada)
busqueda_tramos.fit(X_300, tramo_300)

# Tabla de resultados: C, log-loss media de validación y su desviación típica (la usamos en A4).
resultados_tramos = pd.DataFrame(busqueda_tramos.cv_results_)
resultados_tramos = resultados_tramos[["param_modelo__C", "mean_test_score", "std_test_score"]].copy()

# Cambiamos el signo: de -log-loss a log-loss.
resultados_tramos["mean_test_score"] = -resultados_tramos["mean_test_score"]
print("C elegido:", busqueda_tramos.best_params_["modelo__C"])
print("Log-loss:", round(-busqueda_tramos.best_score_, 3))
resultados_tramos.round(3)

La log-loss baja al reducir `C` desde 100, alcanza un mínimo y vuelve a subir cuando la penalización es muy fuerte. `GridSearchCV` elige el `C` del mínimo.

## A4. La regla del error típico

Cada log-loss media de A3 procede de cinco particiones y tiene incertidumbre. Si la de otro `C` queda a menos de un error típico del mínimo, la validación cruzada no permite distinguirlos.

La **regla del error típico** (tema 2, sección 1.4) elige, entre los `C` cuya log-loss no supera el mínimo más su error típico, el **modelo más sencillo**. Atención al sentido de `C`, que es el inverso de la penalización:

**`C` menor → más penalización → coeficientes más pequeños → modelo más sencillo.**

Por eso la regla elige el `C` **más pequeño** dentro del límite, aunque su log-loss media sea algo mayor que la mínima. La aplicamos en tres pasos.

**Paso 1.** El error típico de cada log-loss media.

In [ ]:
# std_test_score: desviación típica de la log-loss entre las cinco particiones (scikit-learn divide entre 5).
# Error típico de la media: esa desviación dividida por la raíz de 5. Es orientativo, porque las
# cinco log-loss comparten observaciones de ajuste.
resultados_tramos["error_tipico"] = resultados_tramos["std_test_score"] / np.sqrt(5)
resultados_tramos.round(3)

**Paso 2.** El límite de la regla: la log-loss mínima más su error típico.

In [ ]:
# best_index_ es la fila del C de menor log-loss en la tabla de resultados.
fila_minimo = busqueda_tramos.best_index_
limite = resultados_tramos.loc[fila_minimo, "mean_test_score"] + resultados_tramos.loc[fila_minimo, "error_tipico"]
print("Log-loss mínima:", round(resultados_tramos.loc[fila_minimo, "mean_test_score"], 3),
      "| límite:", round(limite, 3))

**Paso 3.** Los `C` con log-loss dentro del límite y, entre ellos, el menor.

In [ ]:
# Filas de la tabla con log-loss que no supera el límite.
dentro_limite = resultados_tramos[resultados_tramos["mean_test_score"] <= limite]

# La regla elige el C menor: el de más penalización.
c_regla = dentro_limite["param_modelo__C"].min()
print("C de menor log-loss:", busqueda_tramos.best_params_["modelo__C"])
print("C de la regla del error típico:", c_regla)

Si el `C` de la regla coincide con el elegido, ningún `C` más penalizado queda cerca del mínimo. Si es menor, el modelo más penalizado tiene una log-loss que no se distingue de la mínima, y la regla lo prefiere por ser más sencillo: coeficientes más pequeños y probabilidades menos extremas.

# Parte B. Actividad

Cada bloque plantea un problema dividido en tareas cortas (T). En cada tarea:

- escribe el código completo en la celda «Escribe aquí tu solución»;
- crea los objetos **con los nombres exactos** que indica el enunciado: la corrección los busca por su nombre;
- adapta el código de la sección de la parte A que se indica.

Si una tarea no te sale, sigue con la siguiente: cada tarea se puntúa por separado.

Las interpretaciones se escriben entre las comillas triples `"""` de su celda. Responde a lo que se pregunta, **con tus cifras**.

## Bloque 1. Otro umbral para el logit binario

Solo un 25 % de los trabajadores tienen un salario alto, y con el umbral 0,5 el modelo deja sin detectar a muchos de ellos. Probamos un umbral más bajo: la proporción de positivos.

**T1.1.** Crea `umbral_b1`, igual a la proporción de positivos en la respuesta `alto_train`, y `pred_b1`, la clase predicha con ese umbral a partir de `prob_alto_cv` (como `pred_05` en A1).

In [ ]:
# T1.1 · umbral_b1 y pred_b1
# Escribe aquí tu solución


**T1.2.** Calcula la sensibilidad `sens_b1` y la precisión `prec_b1` con `pred_b1` (como en A1).

In [ ]:
# T1.2 · sens_b1 y prec_b1
# Escribe aquí tu solución


**T1.3.** Crea la matriz de confusión `matriz_b1` con `pred_b1`, dibújala con `ConfusionMatrixDisplay` y guarda en `fp_b1` el número de falsos positivos (como en A1).

In [ ]:
# T1.3 · matriz_b1 y fp_b1
# Escribe aquí tu solución


**Interpretación 1.** Compara `matriz_b1` con `matriz_05` (A1).

1. Al bajar el umbral de 0,5 a `umbral_b1`, ¿qué casillas de la matriz aumentan y cuáles disminuyen? Da las cifras.
2. Explica con esas casillas por qué sube la sensibilidad y baja la precisión.
3. Si no detectar a un trabajador de salario alto cuesta más que una falsa alarma, ¿cuál de los dos umbrales elegirías? Justifícalo.

In [ ]:
interp_b1 = """
Escribe aquí tu respuesta.
"""

## Bloque 2. Capacidad predictiva del modelo multinomial

Evaluamos con más detalle el logit multinomial de A2.

**T2.1.** Calcula `sens_tramos_b2`, la sensibilidad de cada tramo, con `recall_score` sobre `tramo_train` y `tramo_predicho`. Con tres clases, añade el argumento `average=None`: así devuelve una sensibilidad por tramo, en el orden 0, 1, 2.

In [ ]:
# T2.1 · sens_tramos_b2
# Escribe aquí tu solución


**T2.2.** Un error **extremo** confunde el tramo bajo con el alto. Guarda en `errores_extremos_b2` la suma de los trabajadores de tramo bajo predichos como alto y los de tramo alto predichos como bajo. Léelos de `matriz_tramos` (A2) con índices `[fila, columna]`, como `matriz_05[0, 1]` en A1.

In [ ]:
# T2.2 · errores_extremos_b2
# Escribe aquí tu solución


**T2.3.** Calcula `logloss_b2`, la log-loss del multinomial, con `prob_tramos_cv` (como `logloss_05` en A1). Con tres tramos, `prob_tramos_cv` tiene tres columnas y `log_loss` las usa todas.

In [ ]:
# T2.3 · logloss_b2
# Escribe aquí tu solución


**T2.4.** Calcula `logloss_ref_b2`, la log-loss del clasificador de referencia `referencia` con los tramos. No sirve `tramo_referencia` de A2: contiene tramos predichos (clases) y la log-loss necesita probabilidades. Obtén primero las probabilidades de validación cruzada de `referencia` con `tramo_train`, con `cross_val_predict` y `method="predict_proba"` (como `prob_referencia_cv` en A1), y calcula después con ellas `logloss_ref_b2`.

In [ ]:
# T2.4 · logloss_ref_b2
# Escribe aquí tu solución


**Interpretación 2.**

1. ¿Qué tramo predice peor el modelo? Mira en la matriz de A2 con qué tramos se confunde y explica por qué es el más difícil.
2. Compara `errores_extremos_b2` con los errores entre tramos vecinos (bajo-medio y medio-alto). ¿Por qué hay tan pocos errores extremos?
3. ¿Cuánto mejora la log-loss del modelo a la de la referencia?

In [ ]:
interp_b2 = """
Escribe aquí tu respuesta.
"""

## Bloque 3. Penalización del logit binario con pocas observaciones

Repetimos la búsqueda de A3 con la respuesta binaria en una submuestra de 300 observaciones.

**T3.1.** Crea `X_300` y `alto_300`, los predictores y la respuesta binaria de una submuestra de 300 observaciones del entrenamiento: usa `train_test_split` con `alto_train`, `train_size=300` y `random_state=SEMILLA` (como en A3). Salen las mismas 300 filas que en A3.

In [ ]:
# T3.1 · X_300 y alto_300
# Escribe aquí tu solución


**T3.2.** Crea `busqueda_b3`, un `GridSearchCV` con el modelo `logit` de A1, la malla `MALLA_C`, la métrica `"neg_log_loss"` y las particiones `cv_estratificada`, y ajústalo con `X_300` y `alto_300` (como en A3).

In [ ]:
# T3.2 · busqueda_b3
# Escribe aquí tu solución


**T3.3.** Guarda en `c_elegido_b3` el valor de `C` de menor log-loss (el número que guarda `best_params_["modelo__C"]`, como en A3) y en `logloss_mejor_b3` esa log-loss, que debe ser **positiva**. Construye también la tabla de resultados `resultados_b3`, con la log-loss en positivo y su desviación típica (como `resultados_tramos` en A3): la usarás en la interpretación y en el bloque 4.

In [ ]:
# T3.3 · c_elegido_b3, logloss_mejor_b3 y resultados_b3
# Escribe aquí tu solución


**Interpretación 3.** Usa tu tabla `resultados_b3`.

1. Con 300 observaciones, ¿mejora la penalización la log-loss **en tu muestra**? Compara `logloss_mejor_b3` con la log-loss del `C` mayor de la malla (`C` = 100).
2. ¿Por qué un `C` grande, con poca penalización, puede empeorar la log-loss cuando hay pocas observaciones? ¿Esperarías la misma ganancia con las 7 500 observaciones del entrenamiento?

In [ ]:
interp_b3 = """
Escribe aquí tu respuesta.
"""

## Bloque 4. La regla del error típico

Aplicamos a la búsqueda del bloque 3 la regla de A4.

**T4.1.** Añade a `resultados_b3` la columna `error_tipico` y guarda en `limite_b4` el límite de la regla: la log-loss mínima más su error típico (pasos 1 y 2 de A4).

In [ ]:
# T4.1 · error_tipico y limite_b4
# Escribe aquí tu solución


**T4.2.** Guarda en `c_regla_b4` el `C` de la regla del error típico: el menor `C` con log-loss dentro del límite (paso 3 de A4).

In [ ]:
# T4.2 · c_regla_b4
# Escribe aquí tu solución


**Interpretación 4.**

1. ¿Coinciden `c_elegido_b3` y `c_regla_b4`? Explícalo con `limite_b4` y la log-loss de los `C` de tu tabla.
2. ¿Qué `C` elegirías? Justifícalo. Recuerda: un `C` menor significa más penalización y un modelo más sencillo.

In [ ]:
interp_b4 = """
Escribe aquí tu respuesta.
"""

# Cierre: guarda tus respuestas

1. Reinicia el kernel y ejecuta todas las celdas: en VS Code, botones **Restart** y **Run All** de la barra superior del cuaderno. En Jupyter, el icono ⏩ de la barra (reiniciar el kernel y ejecutar todo). El cuaderno debe ejecutarse de principio a fin.
2. Comprueba que la celda siguiente no señala ninguna respuesta pendiente.
3. Sube a Aula Virtual **dos ficheros**: este cuaderno (`.ipynb`) y `entrega_E1_<tu ID>.json`, que se crea en la misma carpeta que el cuaderno.

**No modifiques esta celda.**

In [ ]:
# Objetos que se corrigen, por su nombre.
NOMBRES = [
    "umbral_b1", "pred_b1", "sens_b1", "prec_b1", "matriz_b1", "fp_b1", "interp_b1",
    "sens_tramos_b2", "errores_extremos_b2", "logloss_b2", "logloss_ref_b2", "interp_b2",
    "alto_300", "c_elegido_b3", "logloss_mejor_b3", "interp_b3",
    "limite_b4", "c_regla_b4", "interp_b4",
]

def a_python(valor):
    # Convierte objetos de numpy y pandas en tipos que admite JSON.
    if hasattr(valor, "tolist"):
        return valor.tolist()
    return valor

respuestas = {nombre: a_python(globals().get(nombre)) for nombre in NOMBRES}
for clave in ["pred_b1", "alto_300"]:
    # De los vectores largos basta con su suma.
    if respuestas[clave] is not None:
        respuestas[clave] = int(np.sum(respuestas[clave]))
respuestas.update({
    "semilla": SEMILLA, "sklearn": sklearn.__version__,
    "matriz_05": matriz_05.tolist(), "matriz_tramos": matriz_tramos.tolist(),
})

fichero = f"entrega_E1_{ID}.json"
with open(fichero, "w", encoding="utf-8") as f:
    json.dump(respuestas, f, ensure_ascii=False, indent=2, default=str)

pendientes = [n for n in NOMBRES if respuestas[n] is None
              or (n.startswith("interp") and "Escribe aquí tu respuesta" in str(respuestas[n]))]
print("Guardado:", fichero)
print("Respuestas pendientes:", pendientes if pendientes else "ninguna")